# Preliminaries

In this assignment, you will implement sampling-based motion planning of the 7DOF Kuka iiwa robot. Specifically, you will separately implement a probabilistic roadmap approach and a rapidly exploring random tree approach, followed by some simple post-processing.

In [0]:
# Run once at the beginning to install pybullet
!pip install pybullet

In [0]:
import pybullet as p
import pybullet_data

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.collections import LineCollection

import networkx as nx
from scipy.spatial import KDTree
from scipy.interpolate import splprep, splev

In [0]:
# Load Plane, Kuka iiwa, and an obstacle
sim_id = p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

plane_id = p.loadURDF("plane.urdf", physicsClientId=sim_id)
robot_id = p.loadURDF("kuka_iiwa/model.urdf", useFixedBase=True, physicsClientId=sim_id)

# Create a tall central wall in front of the robot
wall_shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.05, 0.20, 0.35], physicsClientId=sim_id)
wall_visual = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.05, 0.20, 0.35],
                                  rgbaColor=[0.8, 0.2, 0.2, 0.8], physicsClientId=sim_id)

wall_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=wall_shape,
                            baseVisualShapeIndex=wall_visual,
                            basePosition=[0.55, 0.0, 0.35], physicsClientId=sim_id)

wall2_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=wall_shape,
                             baseVisualShapeIndex=wall_visual,
                             basePosition=[-0.55, 0.0, 0.35], physicsClientId=sim_id)

NUM_JOINTS = 7
JOINT_LIMITS_LOWER = np.array([-3, -2, -3, -2, -3, -2, -3])
JOINT_LIMITS_UPPER = np.array([ 3,  2,  3,  2,  3,  2,  3])

In [0]:
# Robot utility functions

def set_joint_positions(robot_id, q):
    """Sets the robot joints to a given configuration vector q."""
    for i in range(NUM_JOINTS):
        p.resetJointState(robot_id, i, q[i], physicsClientId=sim_id)

def is_valid_configuration(robot_id, q):
    """Checks if a configuration q is within joint limits and collision-free."""
    # 1. Joint limit check
    if np.any(q < JOINT_LIMITS_LOWER) or np.any(q > JOINT_LIMITS_UPPER):
        return False

    # 2. Update robot configuration in simulation
    set_joint_positions(robot_id, q)
    p.performCollisionDetection(physicsClientId=sim_id)

    # 3. Check for contact points with environment or self
    contacts = p.getContactPoints(physicsClientId=sim_id)
    return len(contacts) == 0

In [0]:
# Visualization utility functions

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams['figure.dpi'] = 90
plt.ioff()

def animate_solution(raw_path):
    # Interpolate intermediate steps for smooth video frame playback
    dense_path = []
    for i in range(len(raw_path) - 1):
        segment_dist = np.linalg.norm(raw_path[i+1] - raw_path[i])
        steps = max(int(np.ceil(segment_dist / 0.2)), 1)
        for t in np.linspace(0, 1, steps, endpoint=False):
            dense_path.append(raw_path[i] + t * (raw_path[i+1] - raw_path[i]))
    dense_path.append(raw_path[-1])

    fig, ax = plt.subplots()
    def animate(frame):
        ax.cla()
        for i in range(len(dense_path[frame])):
            p.resetJointState(robot_id, i, dense_path[frame][i], physicsClientId=sim_id)
        showImage(ax)
    anim = animation.FuncAnimation(fig, animate, frames=len(dense_path))
    plt.close(fig)
    return anim


def showImage(ax, cameraPos=[2, 2, 2]):
    viewMatrix = p.computeViewMatrix(
                cameraEyePosition=cameraPos,
                cameraTargetPosition=[0, 0, 0],
                cameraUpVector=[0, 0, 1],
                physicsClientId=sim_id)
    projectionMatrix = p.computeProjectionMatrixFOV(
                fov=60.0,
                aspect=1.0,
                nearVal=0.1,
                farVal=10,
                physicsClientId=sim_id)
    width, height, rgbImg, depthImg, segImg = p.getCameraImage(
                width=512,
                height=512,
                viewMatrix=viewMatrix,
                projectionMatrix=projectionMatrix,
                physicsClientId=sim_id)
    rgbImg = np.array(rgbImg, dtype=np.uint8).reshape((512, 512, 4))
    ax.imshow(rgbImg)

# Part 1: Probabilistic Roadmap

A probabilistic roadmap gives us a coarse approximation of the full 7-dimensional C-space. Once constructed, we can save and use it for multiple different queries. We can also modify or add new nodes and edges to the roadmap as needed.

The following is a plotting function specifically for the PRM.

In [0]:
def plot_prm_graph_2d(graph, path=None, q_start=None, q_goal=None):
    """Plots the PRM roadmap graph and solution path projected onto Joint 1 vs Joint 2.
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    # 1. Extract 2D joint positions (Joint 1 and Joint 2) for all graph nodes
    node_ids = list(graph.nodes())
    node_positions = np.array([graph.nodes[n]['q'][:2] for n in node_ids])
    node_map = {n: graph.nodes[n]['q'][:2] for n in node_ids}

    # 2. Extract edge line segments for high-performance rendering via LineCollection
    edge_segments = []
    for u, v in graph.edges():
        edge_segments.append([node_map[u], node_map[v]])

    lc = LineCollection(edge_segments, colors='lightgray', linewidths=0.7, alpha=0.5, label='Roadmap Edges')
    ax.add_collection(lc)

    # 3. Scatter plot roadmap nodes
    ax.scatter(node_positions[:, 0], node_positions[:, 1],
               c='skyblue', edgecolors='royalblue', s=20, zorder=3, label='Nodes ($C_{free}$)')

    # 4. Plot solution path if available
    if path is not None and len(path) > 0:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 0], path_arr[:, 1],
                color='crimson', linewidth=2.5, marker='o', markersize=4, zorder=4, label='PRM Path')

    # 5. Highlight Start and Goal configurations
    if q_start is not None:
        ax.scatter(q_start[0], q_start[1], color='limegreen', s=150, marker='*',
                   edgecolors='black', zorder=5, label='Start ($q_{start}$)')
    if q_goal is not None:
        ax.scatter(q_goal[0], q_goal[1], color='red', s=150, marker='*',
                   edgecolors='black', zorder=5, label='Goal ($q_{goal}$)')

    # Formatting
    ax.set_xlim([JOINT_LIMITS_LOWER[0] - 0.1, JOINT_LIMITS_UPPER[0] + 0.1])
    ax.set_ylim([JOINT_LIMITS_LOWER[1] - 0.1, JOINT_LIMITS_UPPER[1] + 0.1])
    ax.set_xlabel('Joint 1 Angle $\\theta_1$ (rad)', fontsize=11)
    ax.set_ylabel('Joint 2 Angle $\\theta_2$ (rad)', fontsize=11)
    ax.set_title('PRM Graph Projection (Joint 1 vs Joint 2)', fontsize=13)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.show()
    plt.close()

## 1.1: Local Planner (6 pts)

Recall that the local planner subroutine is used to determine the feasibility of connecting two nodes (configurations) `q1` and `q2` with an edge. A simple method of doing so is to traverse the line segment between them in `step_size` increments, and verify that each increment is collision-free using `is_valid_configuration()`.

Implement `check_edge()` below so that it returns `True` if the edge is valid and `False` otherwise. You are recommended to use either a linear traversal, e.g. from `q1` to `q2`, or a recursive midpoint traversal.

In [0]:
def check_edge(robot_id, q1, q2, step_size=0.03):
    """Validates straight-line edge between q1 and q2 via recursive bisection.

    Returns:
        bool: True if edge is collision-free, False otherwise.
    """
    # TODO
    return False

Run your implementation on the test cases below. Note that passing both does not guarantee that your implementation is correct, so you may have to check it more carefully if debugging is required later on.

In [0]:
q1 = np.array([0.9, 0.9, 0.0, -1.3, 0.0, 0.6, 0.0])
q2 = np.array([0.85, 0.85, 0.0, -1.25, 0.0, 0.55, 0.0])
q3 = np.array([-0.9, 0.9, 0.0, -1.3, 0.0, 0.6, 0.0])

assert check_edge(robot_id, q1, q2) == True, (
        "Test Failed: Expected edge between q1 and q2 to be valid, "
        "but local planner returned False."
    )
print("✓ Test 1 Passed: Collision-free edge correctly validated.")

assert check_edge(robot_id, q1, q3) == False, (
        "Test Failed: Expected direct edge from q1 to q3 to collide with wall, "
        "but local planner returned True."
    )
print("✓ Test 2 Passed: Collision edge correctly detected and rejected.")

## 1.2: Nodes and Edges (12 pts)

To start building a PRM, we need to sample a set of nodes and determine the edge set. First implement `sample_nodes()`, for which we will simply use a uniform random sampler. It should return an array of `n_samples` **valid** configurations, meaning collision-free and within joint limits.

Next, implement `compute_edges()`. To efficiently compute neighborhoods, first instantiate a [KDTree](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.html) with the `nodes`. Then for each node `q`, you can use `KDTree.query()` to find the `k_neighbors` closest neighbors. While we would normally have to use a circle distance metric, joint limits eliminate the "wrap-around" feature of the space, so the default (Euclidean) distance of KDtree will suffice. Given the returned neighborhood, you should apply the following filters:

*   This set will include `q` itself, so you should exclude it. This also means that you should actually be finding the `k_neighbors+1` closest neighbors.
*   Exclude all neighbors that are greater than `max_radius` away from `q`.
*   Exclude all neighbors for which the local planner returns `False`.

For each valid edge with distance `dist` between nodes `q1` and `q2` with indices `i` and `j` in `nodes`, respectively, save it as a tuple `(i, j, dist)`. Return a list of all the valid edges.



In [0]:
def sample_nodes(n_samples, robot_id):
    """Generates configuration samples using uniform pseudorandom sampling.
       Filters generated samples to retain only collision-free configurations.

    Returns:
        np.ndarray: Matrix of shape (n_samples, 7)
    """
    # TODO
    return []

In [0]:
def compute_edges(nodes, k_neighbors=15, max_radius=2.5, step_size=0.03):
    """Finds all valid edges among the given nodes.

    Returns:
        list[tuple]: List of valid edges in the form (i, j, dist).
    """
    # TODO
    return []

To verify your methods, the following code takes the outputs and builds a roadmap graph as a `NetworkX.Graph` object. It also visualizes the graph in $\theta_1$-$\theta_2$ space. However, remember that the full graph lives in the 7-dimensional C-space.

In [0]:
np.random.seed(50)

N_SAMPLES = 1000
K_NEIGHBORS = 20
MAX_RADIUS = 2
STEP_SIZE = 0.02

valid_nodes = sample_nodes(N_SAMPLES, robot_id)
valid_edges = compute_edges(valid_nodes, K_NEIGHBORS, MAX_RADIUS, STEP_SIZE)

prm_graph = nx.Graph()
for i in range(len(valid_nodes)):
    prm_graph.add_node(i, q=valid_nodes[i])

for edge in valid_edges:
    prm_graph.add_edge(edge[0], edge[1], weight=edge[2])

plot_prm_graph_2d(prm_graph)

## 1.3: Path Search (6 pts)

To use the PRM for motion planning, we would first attempt to connect the query points to the roadmap and then perform search. You will implement the second procedure below in `solve_prm_query()`. Given the query node **indices** `q_start` and `q_goal`, perform A* search using [`NetworkX.astar_path()`](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.astar.astar_path.html).

You should use Euclidean distance as the heuristic. In addition, note that it is possible for no path to be found. Be sure to properly handle that case by catch the resultant error.

If a path is found, return it as a list of configurations (tuples). Otherwise, simply return `None`.

In [0]:
def solve_prm_query(roadmap, q_start, q_goal):
    """Given start and goal configurations, solve path query via A* search.

    Returns:
        list[np.ndarray] or None: Ordered joint configurations forming the path, or None if no path.
    """
    # TODO
    return None

Now we have everything that we need. You can run the following code, which will randomly choose two query points on the roadmap and run your pathfinding procedure to attempt to connect them.

If a solution is found, you will see the joint angle trajectories along with the path overlaid on the $\theta_1$-$\theta_2$ projection of the roadmap. Finally, running the cell after this one will generate an animation of the robot following this trajectory.

In [0]:
q_start = np.random.randint(N_SAMPLES)
q_goal = np.random.randint(N_SAMPLES)
raw_path = solve_prm_query(prm_graph, q_start, q_goal)

if raw_path is not None:
    plot_prm_graph_2d(prm_graph, raw_path, valid_nodes[q_start], valid_nodes[q_goal])
else:
    print("No path found to visualize!")

In [0]:
if raw_path is not None:
    anim = animate_solution(raw_path)
    display(anim)

# Part 2: Rapidly Exploring Random Tree

Instead of constructing a general roadmap, a rapidly exploring random tree is used to solve motion planning for a specific pair of query points. As with the PRM, we will first separately implement a couple of subroutines, and then use them to fully implement the algorithm.

In [0]:
def plot_rrt_tree_2d(tree, path, q_start, q_goal):
    """Visualizes the 2D projection (Joint 1 vs Joint 2) of an RRT tree and its solution path.
    """
    # Clear any previous figure buffers to prevent duplicate rendering
    plt.close('all')
    fig, ax = plt.subplots(figsize=(8, 6))

    # 1. Plot all tree edges (light gray lines)
    for parent, child in tree.edges():
        q_parent = tree.nodes[parent]['q']
        q_child = tree.nodes[child]['q']
        ax.plot(
            [q_parent[0], q_child[0]],
            [q_parent[1], q_child[1]],
            color='lightgray',
            linewidth=0.8,
            zorder=1
        )

    # 2. Plot all tree nodes (small blue dots)
    tree_configs = np.array([tree.nodes[n]['q'] for n in tree.nodes()])
    ax.scatter(
        tree_configs[:, 0],
        tree_configs[:, 1],
        c='dodgerblue',
        s=12,
        alpha=0.6,
        edgecolors='none',
        label='Tree Nodes',
        zorder=2
    )

    # 3. Highlight solution path if one exists (bold crimson line with waypoint markers)
    if path is not None and len(path) > 0:
        path_configs = np.array(path)
        ax.plot(
            path_configs[:, 0],
            path_configs[:, 1],
            color='crimson',
            linewidth=2.5,
            linestyle='-',
            marker='o',
            markersize=5,
            label='Solution Path',
            zorder=3
        )

    # 4. Plot Start and Goal configurations
    ax.scatter(
        q_start[0], q_start[1],
        c='lime', edgecolors='black', s=160, marker='*', zorder=4, label=r'Start ($q_{start}$)'
    )
    ax.scatter(
        q_goal[0], q_goal[1],
        c='red', edgecolors='black', s=160, marker='*', zorder=4, label=r'Goal ($q_{goal}$)'
    )

    # Plot formatting
    ax.set_xlabel(r'Joint 1 Angle $\theta_1$ (rad)', fontsize=12)
    ax.set_ylabel(r'Joint 2 Angle $\theta_2$ (rad)', fontsize=12)
    ax.set_title(r'RRT Tree & Solution Path Projection ($\theta_1$ vs $\theta_2$)', fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.show()

## 2.1: Sampling and Nearest Node (6 pts)

Implement `sample_and_nearest()`, which samples a configuration $q_{rand}$ and finds the nearest tree node $q_{near}$ to it. Here, the "tree" is just a list of existing configurations in `tree_configs`.

Let $q_{rand}$ be $q_{goal}$ with probability `goal_bias`. Otherwise, $q_{rand}$ is a random configuration within joint limits.

Find $q_{near}$ in the tree. You can use Euclidean distance between $q_{rand}$ and the existing configurations.

Return a tuple containing the configuration $q_{rand}$ as well as the *index* of $q_{near}$ in `tree_configs`.

In [0]:
def sample_and_nearest(tree_configs, q_goal, goal_bias=0.05):
    """Samples q_rand with goal biasing and identifies the nearest node in the tree.

    Returns:
        tuple: (q_rand, near_idx) where near_idx is the index into tree_configs.
    """
    # TODO
    return None, None

## 2.2: Steering and Collision Checking (8 pts)

Implement `steer_and_check()`, which proposes a new configuration $q_{new}$ to add to a current tree.

Let $q_{new}$ be a configuration on the segment between $q_{near}$ and $q_{rand}$. It should be either $q_{rand}$ itself or a configuration that is a distance of `steer_step` away from $q_{near}$, whichever is closer.

Using `check_edge()` from Part 1, check that the segment between $q_{near}$ and $q_{new}$ is completely collision-free using an increment of `local_step`. If so, return the configuration $q_{new}$; otherwise, return `None`.

In [0]:
def steer_and_check(q_rand, q_near, steer_step=0.20, local_step=0.03):
    """Steers from q_near toward q_rand and validates state and edge collisions.

    Returns:
        np.ndarray or None: Valid q_new array if collision-free; None if invalid.
    """
    # TODO
    return None

The following code builds a full RRT, given a starting and goal configuration. After adding a new node, each iteration also performs a "goal test", in which we attempt to add the goal to the tree if it is sufficiently close to the new node. We also specify `max_iters`, which stops the procedure after tree reaches the given size.

In [0]:
def rrt_solve(
    q_start,
    q_goal,
    max_iters=500,
    steer_step=0.20,
    goal_threshold=0.20,
    goal_bias=0.05,
    local_step=0.03
):
    """Solves motion planning query using basic Rapidly-exploring Random Tree (RRT).

    Returns:
        list[np.ndarray] or None: Path as list of 7D configurations, or None if failed.
    """
    # Pre-check start and goal validity
    if not is_valid_configuration(robot_id, q_start) or not is_valid_configuration(robot_id, q_goal):
        raise ValueError("Start or Goal configuration is in collision or out of joint limits!")

    # Graph structure to hold tree nodes and parent references
    tree = nx.DiGraph()
    tree.add_node(0, q=q_start)
    tree_configs = [q_start]  # Keep array list for fast vector distance lookups

    for i in range(1, max_iters + 1):
        q_rand, near_idx = sample_and_nearest(tree_configs, q_goal, goal_bias)
        q_near = tree_configs[near_idx]
        q_new = steer_and_check(q_rand, q_near, steer_step, local_step)
        if q_new is None:
            continue

        new_idx = len(tree_configs)
        tree.add_node(new_idx, q=q_new)
        tree.add_edge(near_idx, new_idx)
        tree_configs.append(q_new)

        # Check if q_new is within goal threshold
        dist_to_goal = np.linalg.norm(q_goal - q_new)
        if dist_to_goal <= goal_threshold:
            # Attempt final connection from q_new to q_goal
            if check_edge(robot_id, q_new, q_goal, local_step):
                goal_node_id = len(tree_configs)
                tree.add_node(goal_node_id, q=q_goal)
                tree.add_edge(new_idx, goal_node_id)

                # Backtrack path from root (node 0) to goal_node_id
                path_nodes = nx.shortest_path(tree, source=0, target=goal_node_id)
                print(f"RRT succeeded in {i} iterations! Tree size: {len(tree_configs)} nodes.")
                return tree, [tree.nodes[n]['q'] for n in path_nodes]

    print(f"RRT reached max iterations ({max_iters}) without reaching the goal.")
    return tree, None

In [0]:
np.random.seed(50)
q_start = np.array([ 1.0,  0.8,  0.0, -1.2,  0.0,  0.6,  0.0])
q_goal  = np.array([-1.0,  0.8,  0.0, -1.2,  0.0,  0.6,  0.0])
tree, path = rrt_solve(q_start, q_goal)
plot_rrt_tree_2d(tree, path, q_start, q_goal)

In [0]:
if path is not None:
    anim = animate_solution(path)
    display(anim)

# Part 3: Path Post-Processing

You have probably seen that the raw paths produced by PRM and RRT are piecewise-linear, jerky, and (at times) redundant due to random sampling. To transform these paths into realistic command trajectories, we will attempt to 1) shorten them geometrically and 2) apply spline interpolation.

## 3.1: Randomized Shortcutting (6 pts)

Given a path, we can strip out unnecessary waypoints by attempting to connect non-adjacent nodes. Implement `shortcut_path()` to make a copy of the given `path` and iteratively modify it. Randomly sample pairs of non-adjacent path indices and use `check_edge()` to verify if the corresponding configurations can be connected. If so, update the path and drop the intermediate nodes.

Return the (possibly) shortened path after running this `num_iterations` times.

In [0]:
def shortcut_path(path, num_iterations=100, local_step=0.03):
    """Applies randomized shortcutting to remove redundant waypoints from a C-space path.

    Returns:
        list[np.ndarray]: Geometrically shortened waypoint path.
    """
    # TODO
    return None

You can try out the method by applying it to the RRT result from above and displaying the shortcutted path.

In [0]:
shortcut_waypoints = shortcut_path(path)
plot_rrt_tree_2d(tree, shortcut_waypoints, q_start, q_goal)

## 3.2: Linear Densification (6 pts)

The shortcutted path is representative of the robot's desired trajectory, but we need to "densify" it for controller commands (e.g., to compute desired velocities or torques). Implement `densify_linear_path()` so that it takes in a sparse set of waypoints and constructs an interpolated path consisting of `num_samples` points.

One way to do this is to first compute the proportion of the total distance captured by each piecewise segment. You can then use the same proportion of `num_samples` points to interpolate the corresponding segment.

In [0]:
def densify_linear_path(waypoints, num_samples=100):
    """Generates a dense, linearly interpolated trajectory along sparse waypoints.

    Returns:
        np.ndarray: Array of shape (num_samples, 7) representing the dense path.
    """
    # TODO
    return None

You can try out the method by applying it to the shortcutted path from above and displaying the densified path.

In [0]:
dense_linear_path = densify_linear_path(shortcut_waypoints)
plot_rrt_tree_2d(tree, dense_linear_path, q_start, q_goal)

## 3.3: Spline Interpolation

Now we have an efficient path consisting of dense waypoints in linear segments. Following this path would require to the robot to come to a stop at each waypoint to avoid infinite acceleration. To address this, we can use spline interpolation of the original waypoints to construct a smooth trajectory.

In practice, we would need to check that the spline is valid at all points, since it will contain new points that were not previously checked. If the spline contains a collision, we would have to use the dense piecewise-linear path as the fallback option.

You do not need to implement anything here; simply run the interpolation routine to see the final results.

In [0]:
def generate_smooth_spline(path, num_samples=100, k=3):
    """Fits a B-spline through waypoints and validates safety in PyBullet.

    Returns:
        np.ndarray: Collision-free trajectory array of shape (num_samples, 7).
    """
    k_eff = min(k, len(path) - 1)
    configs = np.array(path).T  # Shape: (7, N)
    tck, _ = splprep(configs, s=0, k=k_eff)

    u_fine = np.linspace(0, 1, num_samples)
    smooth_trajectory = np.array(splev(u_fine, tck)).T  # Shape: (num_samples, 7)
    return smooth_trajectory

In [0]:
smooth_trajectory = generate_smooth_spline(shortcut_waypoints)
plot_rrt_tree_2d(tree, smooth_trajectory, q_start, q_goal)

In [0]:
if smooth_trajectory is not None:
    anim = animate_solution(smooth_trajectory)
    display(anim)